# Prapemrosesan Data & Augmentasi Tingkat Lanjut

In [1]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==========================================
# 1. KONFIGURASI JALUR & HIPERPARAMETER
# ==========================================
BASE_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026"
TRAIN_DIR = os.path.join(BASE_PATH, "train")

IMG_SIZE = 384 
BATCH_SIZE = 8

# ==========================================
# 2. PEMBUATAN DATAFRAME & STRATIFIED SPLIT
# ==========================================
class_names = sorted(os.listdir(TRAIN_DIR))
label2id = {class_name: i for i, class_name in enumerate(class_names)}
id2label = {i: class_name for class_name, i in label2id.items()}

data = []
for class_name in class_names:
    class_dir = os.path.join(TRAIN_DIR, class_name)
    if os.path.isdir(class_dir):
        for img_name in os.listdir(class_dir):
            data.append({
                "image_path": os.path.join(class_dir, img_name),
                "label": label2id[class_name]
            })

df_all = pd.DataFrame(data)

train_df, valid_df = train_test_split(
    df_all, test_size=0.2, stratify=df_all['label'], random_state=42
)

# ==========================================
# 3. PIPELINE AUGMENTASI (ALBUMENTATIONS)
# ==========================================
train_transforms = A.Compose([
    # Menggunakan 'size' untuk RandomResizedCrop sesuai versi terbaru
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.3),
    
    # Mengganti ShiftScaleRotate dengan Affine untuk menghindari warning
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-15, 15), p=0.5),
    
    # Menyesuaikan nama parameter CoarseDropout untuk versi terbaru
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, p=0.5),
    
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

valid_transforms = A.Compose([
    # Menggunakan 'height' dan 'width' secara eksplisit untuk Resize agar tidak error
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 4. KELAS PYTORCH DATASET
# ==========================================
class SpoofingDataset(Dataset):
    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = cv2.imread(img_path)
        
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']

        label = self.df.loc[idx, 'label']
        return image, torch.tensor(label, dtype=torch.long)

# ==========================================
# 5. INISIALISASI DATALOADER
# ==========================================
train_dataset = SpoofingDataset(train_df, transforms=train_transforms)
valid_dataset = SpoofingDataset(valid_df, transforms=valid_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Prapemrosesan Selesai!")
print(f"Total Data Latih    : {len(train_dataset)} gambar")
print(f"Total Data Validasi : {len(valid_dataset)} gambar")
print(f"Kamus Label         : {id2label}")

Prapemrosesan Selesai!
Total Data Latih    : 1321 gambar
Total Data Validasi : 331 gambar
Kamus Label         : {0: 'fake_mannequin', 1: 'fake_mask', 2: 'fake_printed', 3: 'fake_screen', 4: 'fake_unknown', 5: 'realperson'}


# Membangun Arsitektur Swin

In [2]:
import torch
import torch.nn as nn
import timm

# ==========================================
# 1. KONFIGURASI PERANGKAT (GPU / CPU)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan perangkat: {device}")

# ==========================================
# 2. INISIALISASI MODEL SWIN TRANSFORMER
# ==========================================
MODEL_NAME = 'swin_base_patch4_window12_384.ms_in22k'
NUM_CLASSES = 6 

print(f"Mengunduh dan membangun model: {MODEL_NAME}...")

# REVISI: Menambahkan Drop Rate dan Attention Drop Rate.
# Ini adalah kunci untuk menembus skor tinggi pada dataset kecil.
model = timm.create_model(
    MODEL_NAME, 
    pretrained=True, 
    num_classes=NUM_CLASSES,
    drop_rate=0.3,       # Mematikan 30% jalur saraf di lapisan akhir
    attn_drop_rate=0.2   # Mematikan 20% matriks attention secara acak
)

model = model.to(device)

# ==========================================
# 3. UJI COBA MODEL (SANITY CHECK)
# ==========================================
dummy_input = torch.randn(4, 3, 384, 384).to(device)

with torch.no_grad():
    dummy_output = model(dummy_input)

print("Model berhasil dibangun dengan regulerisasi Dropout!")
print(f"Bentuk input (Batch, Channel, Height, Width) : {dummy_input.shape}")
print(f"Bentuk output (Batch, Num_Classes)           : {dummy_output.shape}")

Menggunakan perangkat: cuda
Mengunduh dan membangun model: swin_base_patch4_window12_384.ms_in22k...


model.safetensors:   0%|          | 0.00/451M [00:00<?, ?B/s]

Model berhasil dibangun dengan regulerisasi Dropout!
Bentuk input (Batch, Channel, Height, Width) : torch.Size([4, 3, 384, 384])
Bentuk output (Batch, Num_Classes)           : torch.Size([4, 6])


# Strategi Pelatihan (Warm-up & Loss)

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.metrics import f1_score
from tqdm import tqdm
import numpy as np

# ==========================================
# 1. DEFINISI FOCAL LOSS
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma 
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        F_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return torch.mean(F_loss)
        return F_loss

# ==========================================
# 2. INISIALISASI OPTIMIZER & SCALER
# ==========================================
EPOCHS = 30  
LEARNING_RATE = 1e-5 
PATIENCE = 10 
ACCUMULATION_STEPS = 4 # Menyeimbangkan Batch Size 4 menjadi 16

criterion = FocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Inisialisasi GradScaler untuk mencegah Out of Memory
scaler = torch.cuda.amp.GradScaler()

# ==========================================
# 3. FUNGSI PELATIHAN (TRAINING LOOP)
# ==========================================
best_val_f1 = 0.0
patience_counter = 0

print("Memulai Pelatihan Swin Transformer (AMP + Gradient Accumulation)...")

for epoch in range(EPOCHS):
    # --- FASE TRAINING ---
    model.train()
    train_loss = 0.0
    optimizer.zero_grad() 
    
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for batch_idx, (images, labels) in enumerate(train_bar):
        images, labels = images.to(device), labels.to(device)
        
        # Menggunakan mixed precision
        with torch.cuda.amp.autocast():
            outputs = model(images) 
            loss = criterion(outputs, labels) 
            loss = loss / ACCUMULATION_STEPS 
        
        # Scaling gradien agar tidak hilang saat menggunakan 16-bit
        scaler.scale(loss).backward() 
        
        if ((batch_idx + 1) % ACCUMULATION_STEPS == 0) or (batch_idx + 1 == len(train_loader)):
            scaler.step(optimizer) 
            scaler.update()
            optimizer.zero_grad() 
        
        train_loss += loss.item() * ACCUMULATION_STEPS * images.size(0)
        train_bar.set_postfix(loss=(loss.item() * ACCUMULATION_STEPS))
        
    avg_train_loss = train_loss / len(train_loader.dataset)
    
    # --- FASE VALIDASI ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    
    val_bar = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
    with torch.no_grad(): 
        for images, labels in val_bar:
            images, labels = images.to(device), labels.to(device)
            
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(valid_loader.dataset)
    val_macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"\nHasil Epoch {epoch+1}:")
    print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
    
    scheduler.step()
    
    # --- SIMPAN MODEL TERBAIK & EARLY STOPPING ---
    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save(model.state_dict(), 'best_swin_model.pth')
        print("Model membaik! Disimpan ke 'best_swin_model.pth'")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"Model tidak membaik. Kesabaran: {patience_counter}/{PATIENCE}")
        
    if patience_counter >= PATIENCE:
        print("\nEarly Stopping dipicu! Pelatihan dihentikan untuk mencegah overfitting.")
        break

print(f"\nPelatihan Selesai! Skor Macro F1 Validasi Terbaik: {best_val_f1:.4f}")

/tmp/ipykernel_55/1175725777.py:41: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Memulai Pelatihan Swin Transformer (AMP + Gradient Accumulation)...


Epoch 1/30 [Train]:   0%|          | 0/166 [00:00<?, ?it/s]/tmp/ipykernel_55/1175725777.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Valid]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/1175725777.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Valid]: 100%|██████████| 42/42 [00:07<00:00,  5.51it/s]



Hasil Epoch 1:
Train Loss: 0.9857 | Val Loss: 0.6240 | Val Macro F1: 0.5863
Model membaik! Disimpan ke 'best_swin_model.pth'


Epoch 2/30 [Train]:   0%|          | 0/166 [00:00<?, ?it/s]/tmp/ipykernel_55/1175725777.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/30 [Valid]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/1175725777.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 2/30 [Valid]: 100%|██████████| 42/42 [00:07<00:00,  6.00it/s]


...


Hasil Epoch 15:
Train Loss: 0.1192 | Val Loss: 0.3774 | Val Macro F1: 0.7797
Model tidak membaik. Kesabaran: 8/10


Epoch 16/30 [Train]:   0%|          | 0/166 [00:00<?, ?it/s]/tmp/ipykernel_55/1175725777.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 16/30 [Valid]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/1175725777.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 16/30 [Valid]: 100%|██████████| 42/42 [00:07<00:00,  5.94it/s]



Hasil Epoch 16:
Train Loss: 0.1101 | Val Loss: 0.3752 | Val Macro F1: 0.7604
Model tidak membaik. Kesabaran: 9/10


Epoch 17/30 [Train]:   0%|          | 0/166 [00:00<?, ?it/s]/tmp/ipykernel_55/1175725777.py:62: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 17/30 [Valid]:   0%|          | 0/42 [00:00<?, ?it/s]/tmp/ipykernel_55/1175725777.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 17/30 [Valid]: 100%|██████████| 42/42 [00:07<00:00,  5.92it/s]


Hasil Epoch 17:
Train Loss: 0.1035 | Val Loss: 0.3876 | Val Macro F1: 0.7756
Model tidak membaik. Kesabaran: 10/10

Early Stopping dipicu! Pelatihan dihentikan untuk mencegah overfitting.

Pelatihan Selesai! Skor Macro F1 Validasi Terbaik: 0.8069


# Kalibrasi Probabilitas (Thresholding)

In [4]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize
import warnings

# Mengabaikan warning scipy agar output terminal tetap bersih
warnings.filterwarnings("ignore")

# ==========================================
# 1. MUAT MODEL TERBAIK & PERSIAPAN DATA
# ==========================================
print("Memuat bobot model terbaik...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load('best_swin_model.pth', map_location=device))
model.eval()

all_probs = []
all_labels = []

print("Mengekstrak probabilitas mentah dari Set Validasi...")
with torch.no_grad():
    for images, labels in valid_loader: 
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1) 
        
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# ==========================================
# 2. EVALUASI SEBELUM KALIBRASI
# ==========================================
preds_before = np.argmax(all_probs, axis=1)
f1_before = f1_score(all_labels, preds_before, average='macro')

print("\n--- SEBELUM KALIBRASI ---")
print(f"Macro F1-Score: {f1_before:.5f}")
# Menampilkan detail per kelas untuk analisis kelemahan model
print(classification_report(all_labels, preds_before, target_names=class_names, digits=4))

# ==========================================
# 3. PROSES OPTIMASI THRESHOLD
# ==========================================
def optimize_f1(weights):
    weighted_probs = all_probs * weights
    calibrated_preds = np.argmax(weighted_probs, axis=1)
    return -1.0 * f1_score(all_labels, calibrated_preds, average='macro')

initial_weights = [1.0] * 6 
# Membatasi bobot agar algoritma tidak memberikan nilai yang tidak masuk akal
bounds = [(0.1, 3.0)] * 6 

print("\nMemulai optimasi kalibrasi probabilitas (Metode: Powell)...")
result = minimize(optimize_f1, initial_weights, method='Powell', bounds=bounds)

best_weights = result.x

# ==========================================
# 4. EVALUASI SETELAH KALIBRASI
# ==========================================
weighted_probs_after = all_probs * best_weights
preds_after = np.argmax(weighted_probs_after, axis=1)
f1_after = f1_score(all_labels, preds_after, average='macro')

print("\n--- SETELAH KALIBRASI ---")
print(f"Macro F1-Score: {f1_after:.5f}")
print(f"Peningkatan   : {f1_after - f1_before:.5f}")
print(classification_report(all_labels, preds_after, target_names=class_names, digits=4))

print("\nSimpan array bobot ini untuk dipakai di Test Set (Tahap 5):")
print(f"best_weights = {list(best_weights)}")

Memuat bobot model terbaik...
Mengekstrak probabilitas mentah dari Set Validasi...

--- SEBELUM KALIBRASI ---
Macro F1-Score: 0.80689
                precision    recall  f1-score   support

fake_mannequin     0.8298    0.8667    0.8478        45
     fake_mask     0.8431    0.7679    0.8037        56
  fake_printed     0.6522    0.6522    0.6522        23
   fake_screen     0.8409    0.8043    0.8222        46
  fake_unknown     0.8594    0.8088    0.8333        68
    realperson     0.8431    0.9247    0.8821        93

      accuracy                         0.8308       331
     macro avg     0.8114    0.8041    0.8069       331
  weighted avg     0.8311    0.8308    0.8299       331


Memulai optimasi kalibrasi probabilitas (Metode: Powell)...

--- SETELAH KALIBRASI ---
Macro F1-Score: 0.83446
Peningkatan   : 0.02757
                precision    recall  f1-score   support

fake_mannequin     0.8723    0.9111    0.8913        45
     fake_mask     0.8000    0.8571    0.8276        5

# Prediksi TTA & Pengumpulan (Submission)

In [5]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. PERSIAPAN JALUR & TEMPLATE SUBMISSION
# ==========================================
TEST_DIR = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/test"
SAMPLE_SUB_PATH = "/kaggle/input/competitions/data-analytics-competition-dac-find-it-2026/samplesubmission.csv"

sub_df = pd.read_csv(SAMPLE_SUB_PATH)

class_names = ['fake_mannequin', 'fake_mask', 'fake_printed', 'fake_screen', 'fake_unknown', 'realperson']
id2label = {i: class_name for i, class_name in enumerate(class_names)}

IMG_SIZE = 384
# Diturunkan menjadi 8 agar tidak terjadi Out of Memory saat prediksi
BATCH_SIZE = 8 

test_transforms = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# ==========================================
# 2. DATASET ANTI-CRASH (DETEKSI OTOMATIS)
# ==========================================
class TestDatasetTTA(Dataset):
    def __init__(self, df, test_dir, transforms):
        self.df = df
        self.test_dir = test_dir
        self.transforms = transforms
        
        self.existing_files = {}
        if os.path.exists(test_dir):
            for f in os.listdir(test_dir):
                name_without_ext = os.path.splitext(f)[0]
                self.existing_files[name_without_ext] = f

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = str(self.df.iloc[idx]['id'])
        img_id_clean = os.path.splitext(img_id)[0] 
        
        image = None
        
        if img_id_clean in self.existing_files:
            img_path = os.path.join(self.test_dir, self.existing_files[img_id_clean])
            image = cv2.imread(img_path) 
            
        if image is None:
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # TTA 1: Gambar Asli
        augmented_orig = self.transforms(image=image)
        img_orig = augmented_orig['image']
        
        # TTA 2: Gambar Dibalik secara Horizontal
        image_flipped = cv2.flip(image, 1) 
        augmented_flip = self.transforms(image=image_flipped)
        img_flip = augmented_flip['image']

        return img_id, img_orig, img_flip

test_dataset = TestDatasetTTA(sub_df, TEST_DIR, transforms=test_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ==========================================
# 3. EKSEKUSI PREDIKSI (INFERENCE)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# REVISI: Memasukkan bobot kalibrasi dari hasil Tahap 4
best_weights = np.array([
    1.2917031201343783, 
    1.1086726156966198, 
    0.7846563764554524, 
    0.7846559645046248, 
    1.0655135128696007, 
    0.7068118913219787
]) 

all_predictions = []

print(f"Memulai prediksi {len(sub_df)} data dengan TTA...")

with torch.no_grad():
    for img_ids, imgs_orig, imgs_flip in tqdm(test_loader, desc="Menebak Data"):
        imgs_orig = imgs_orig.to(device)
        imgs_flip = imgs_flip.to(device)

        # Prediksi dari kedua variasi gambar
        out_orig = model(imgs_orig)
        prob_orig = F.softmax(out_orig, dim=1)

        out_flip = model(imgs_flip)
        prob_flip = F.softmax(out_flip, dim=1)

        # Rata-rata probabilitas TTA
        prob_avg = (prob_orig + prob_flip) / 2.0
        prob_avg = prob_avg.cpu().numpy()

        # Terapkan bobot kalibrasi dari algoritma Powell
        weighted_probs = prob_avg * best_weights
        final_preds = np.argmax(weighted_probs, axis=1)

        all_predictions.extend(final_preds)

# ==========================================
# 4. PENYIMPANAN KE TEMPLATE
# ==========================================
print("Menyusun file submission...")

final_labels = [id2label[pred] for pred in all_predictions]

sub_df['label'] = final_labels
sub_df.to_csv('submission.csv', index=False)

print("Selesai! File 'submission.csv' berhasil dibuat.")
print(sub_df.head())

Memulai prediksi 404 data dengan TTA...


Menebak Data: 100%|██████████| 51/51 [00:36<00:00,  1.40it/s]

Menyusun file submission...
Selesai! File 'submission.csv' berhasil dibuat.
         id         label
0  test_001   fake_screen
1  test_002   fake_screen
2  test_003     fake_mask
3  test_004    realperson
4  test_005  fake_printed


In [6]:
#0,69286